# 08 — WWF LiDAR × Musanga V3_1
Explore the `gs://carbon-maps` DRC survey: 216 georeferenced plots, matching RGB mosaics, LAS clouds, CHM, DTM and DSM. This notebook is **read-only** on GCS and does not modify V3_1 model products. Run in Colab top to bottom. Large imagery is not downloaded automatically.

In [ ]:
# 00 — Install dependencies
%pip -q install google-cloud-storage geopandas shapely pyogrio rasterio folium earthengine-api geemap fsspec gcsfs

In [ ]:
# 01 — Authenticate Google Cloud, mount Drive and initialize Earth Engine
from google.colab import auth, drive
from pathlib import Path
import os
auth.authenticate_user()
drive.mount('/content/drive', force_remount=False)
EE_PROJECT = os.environ.get('EE_PROJECT', '').strip()  # specify your registered GEE Cloud project if necessary
import ee
try:
    if EE_PROJECT: ee.Initialize(project=EE_PROJECT)
    else: ee.Initialize()
    print('Earth Engine ready')
except Exception as e:
    print('GEE not initialized (not required for LiDAR exploration):', repr(e))
    print('Set EE_PROJECT and rerun this cell to enable satellite comparisons.')
MUSANGA_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/Musanga')
WORK_DIR = MUSANGA_ROOT/'V3_1'/'NB08_WWF_Lidar'
WORK_DIR.mkdir(parents=True,exist_ok=True)
print('Outputs:',WORK_DIR)

In [ ]:
# 02 — Bucket inventory (metadata only)
from google.cloud import storage
import pandas as pd, re
from pathlib import PurePosixPath
BUCKET='carbon-maps'
client=storage.Client()
rows=[]
for b in client.list_blobs(BUCKET):
    if b.name.endswith('/'): continue
    m=re.search(r'(UTM\d{1,2}[NS])[_/-]?(?:Plot|plot)[_-]?(\d+)',b.name,re.I)
    plot_id=f'{m.group(1).upper()}_Plot{int(m.group(2))}' if m else None
    rows.append(dict(path=b.name,extension=PurePosixPath(b.name).suffix.lower(),size_GB=(b.size or 0)/1e9,plot_id=plot_id,updated=str(b.updated)))
df=pd.DataFrame(rows)
df.to_csv(WORK_DIR/'carbon_maps_inventory.csv',index=False)
print('Files:',len(df),'GB:',round(df.size_GB.sum(),1))
display(df.groupby('extension').agg(files=('path','size'),GB=('size_GB','sum')).sort_values('GB',ascending=False).head(20))

In [ ]:
# 03 — Match WWF LiDAR products to plot IDs
patterns={
 'RGB':r'^001_Lidar_Rasters/RGB_mosaic/.*_RGB\.tif$',
 'CHM':r'^001_Lidar_Rasters/Lidar_2meter/CHM/.*\.tif$',
 'DTM':r'^001_Lidar_Rasters/Lidar_2meter/DTM/.*\.tif$',
 'DSM':r'^001_Lidar_Rasters/Lidar_2meter/DSM/.*\.tif$',
 'LAS':r'^00-raw-data/Plots/.*\.las$',
 'ECW':r'^00-raw-data/Plots/.*\.ecw$'}
products=pd.DataFrame({'plot_id':sorted(df.plot_id.dropna().unique())})
for kind,pattern in patterns.items():
    counts=df.loc[df.path.str.contains(pattern,case=False,regex=True,na=False)].groupby('plot_id').size()
    products[kind+'_n']=products.plot_id.map(counts).fillna(0).astype(int)
display(products.head(20))
print('Plots with RGB:',(products.RGB_n>0).sum())
products.to_csv(WORK_DIR/'wwf_plot_product_inventory.csv',index=False)

In [ ]:
# 04 — Download only small plot GIS sidecars; do not read broken Ferry_Lines geometry
import tempfile
import geopandas as gpd
from pathlib import Path
prefix='003_GIS_Layers/Lidar_Plots/'
local=Path(tempfile.mkdtemp(prefix='wwf_gis_'))
for b in client.list_blobs(BUCKET,prefix=prefix):
    if b.name.lower().endswith(('.shp','.shx','.dbf','.prj','.cpg','.qpj')):
        b.download_to_filename(str(local/Path(b.name).name))
shps=list(local.glob('*.shp'))
assert shps, 'No plot shapefile found'
plots=gpd.read_file(shps[0]).to_crs('EPSG:4326')
plots=plots[plots.geometry.notna() & ~plots.geometry.is_empty].copy()
plots['geometry']=plots.geometry.make_valid()
plots['plot_id']=plots.Name.astype(str).str.strip()
plots['utm_zone']=plots.plot_id.str.extract(r'^(UTM\d{1,2}[NS])',expand=False)
plots=plots.merge(products,on='plot_id',how='left',validate='one_to_one')
print('Features:',len(plots),'Unique plots:',plots.plot_id.nunique())
display(plots.groupby('utm_zone').agg(plots=('plot_id','size'),rgb=('RGB_n',lambda x:(x>0).sum())))
plots.to_file(WORK_DIR/'wwf_lidar_plots.gpkg',driver='GPKG')

In [ ]:
# 05 — Interactive OSM / Esri imagery plot map
import folium
from folium.plugins import Fullscreen
from IPython.display import display
m=folium.Map(location=[-2.5,23.5],zoom_start=6,tiles='OpenStreetMap',control_scale=True)
folium.TileLayer(tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',attr='Esri World Imagery',name='Esri satellite',overlay=False,show=False).add_to(m)
colors={'UTM33S':'#9467bd','UTM34S':'#e3771c','UTM34N':'#2ca02c','UTM35S':'#d62728','UTM35N':'#1f77b4','UTM36N':'#8c564b'}
foot=folium.FeatureGroup(name='Survey footprints',show=False)
folium.GeoJson(plots[['plot_id','geometry']].to_json(),style_function=lambda f:{'color':'#276749','weight':1,'fillOpacity':.12},tooltip=folium.GeoJsonTooltip(fields=['plot_id'])).add_to(foot)
foot.add_to(m)
for zone,group in plots.groupby('utm_zone'):
    layer=folium.FeatureGroup(name=f'{zone}: {len(group)}')
    for _,r in group.iterrows():
        pt=r.geometry.representative_point()
        counts={k:int(r.get(k+'_n',0) or 0) for k in ('RGB','LAS','CHM','DTM','DSM')}
        html='<b>'+r.plot_id+'</b><br>'+ '<br>'.join(f'{k}: {v}' for k,v in counts.items())
        col=colors.get(zone,'gray')
        folium.CircleMarker([pt.y,pt.x],radius=5,color=col,weight=2,fill=True,fill_color=col if counts['RGB'] else 'white',fill_opacity=.9,tooltip=r.plot_id,popup=html).add_to(layer)
    layer.add_to(m)
w,s,e,n=plots.total_bounds
m.fit_bounds([[s,w],[n,e]])
Fullscreen().add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
m.save(str(WORK_DIR/'wwf_lidar_survey_map.html'))
display(m)

In [ ]:
# 06 — Compute nearest LiDAR plots once verified Musanga UAV ROI coordinates are provided
# Enter coordinates from V3_1 ROI shapefiles; do not substitute town centres.
MUSANGA_SITES={}  # e.g. {'Yangambi': (lat,lon), 'Yoko': (lat,lon)}
from pyproj import Geod
geod=Geod(ellps='WGS84')
if MUSANGA_SITES:
    table=[]
    for name,(lat,lon) in MUSANGA_SITES.items():
        for _,r in plots.iterrows():
            c=r.geometry.representative_point()
            _,_,dist=geod.inv(lon,lat,c.x,c.y)
            table.append(dict(site=name,plot_id=r.plot_id,distance_km=dist/1000,RGB_n=r.RGB_n,LAS_n=r.LAS_n,CHM_n=r.CHM_n))
    near=pd.DataFrame(table).sort_values('distance_km').groupby('site').head(10)
    display(near)
    near.to_csv(WORK_DIR/'nearest_musanga_wwf_plots.csv',index=False)
else: print('Enter georeferenced UAV ROI coordinates to calculate nearest plots.')

In [ ]:
# 07 — Inspect one RGB GeoTIFF remotely without transferring the entire mosaic
import rasterio
sample=df[df.path.str.contains(patterns['RGB'],case=False,regex=True,na=False)].iloc[0]
uri='/vsigs/'+BUCKET+'/'+sample.path
print('Raster:',sample.path,'Size (GB):',round(sample.size_GB,3))
try:
    with rasterio.open(uri) as src:
        print('CRS:',src.crs,'Size:',src.width,src.height,'Bands:',src.count)
        print('Pixel size:',src.res,'Bounds:',src.bounds,'Overviews:',src.overviews(1))
except Exception as exc:
    print('GDAL virtual GCS read unavailable:',repr(exc))
    print('Try a selected small CHM with gsutil cp after checking object size.')

## Follow-up experiments
1. Compare RGB and LiDAR acquisition dates, CRS, pixel resolutions, and horizontal alignment.
2. Overlay actual V3_1 UAV ROI geometries to identify survey overlap.
3. Inspect small matching RGB and CHM windows before selecting LAS subsets.
4. Explore canopy heights, gaps, and crown metrics, and eventually link them to Musanga maps and Sentinel-2/GEE. Avoid treating historical and contemporary observations as simultaneous.